# 04 Data Types

Convert employee data into appropriate data types while preserving existing values and missing information.

## 1. Preparation

Load the processed dataset containing 95,000 employee records.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/03_string_formatting.csv")
df_clean = df.copy()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 95,000
Columns: 32


## 2. Detection

Review column types, missing values, and unique values to identify fields that need conversion.

In [2]:
type_report = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing": df.isna().sum().values,
    "unique": df.nunique(dropna=True).values,
    "example": [
        df[col].dropna().iloc[0] if df[col].notna().any() else None
        for col in df.columns
    ]
})

display(type_report)

,column,dtype,missing,unique,example
0,employee_id,object,0,95000,E145770
1,full_name,object,0,68153,Kenneth Alexander
2,source_dataset,object,0,3,survey_response_2024
3,age,float64,2877,49,43.0
4,gender,object,2865,5,Female
5,ethnicity,object,23796,8,Black or African American
6,education_level,object,4656,5,Associate Degree
7,years_experience,float64,937,41,25.0
8,tenure_years,float64,0,17,0.0
9,department,object,0,12,Finance


## 3. Conversion

Convert whole-number fields to nullable integers, hire dates to datetime, and text fields to strings.

Keep salary and performance scores as floats and missing-value flags as booleans.

In [3]:
integer_columns = [
    "age",
    "years_experience",
    "tenure_years",
    "age_predicted",
    "years_experience_predicted",
    "performance_rating_ordinal"
]

for column in integer_columns:
    values = df_clean[column].dropna()
    assert (values % 1 == 0).all()
    df_clean[column] = df_clean[column].astype("Int64")

df_clean["hire_date"] = pd.to_datetime(
    df_clean["hire_date"],
    format="%Y-%m-%d"
)

text_columns = df_clean.select_dtypes(include="object").columns

df_clean[text_columns] = df_clean[text_columns].astype("string")

assert df_clean.isna().sum().equals(df.isna().sum())

changes = pd.DataFrame({
    "before": df.dtypes.astype(str),
    "after": df_clean.dtypes.astype(str)
})

display(changes[changes["before"] != changes["after"]])

,before,after
employee_id,object,string
full_name,object,string
source_dataset,object,string
age,float64,Int64
gender,object,string
ethnicity,object,string
education_level,object,string
years_experience,float64,Int64
tenure_years,float64,Int64
department,object,string


Education level and pay band are ordinal variables.

I'll define their natural order while keeping the original category labels.

In [4]:
education_order = [
    "High School",
    "Associate Degree",
    "Bachelor's",
    "Master's",
    "PhD"
]

pay_band_order = sorted(
    df_clean["pay_band"].dropna().unique(),
    key=lambda x: int(x.split()[-1])
)

assert df_clean["education_level"].dropna().isin(education_order).all()
assert all(pd.Series(pay_band_order).str.fullmatch(r"Band \d+"))

df_clean["education_level"] = pd.Categorical(
    df_clean["education_level"],
    categories=education_order,
    ordered=True
)

df_clean["pay_band"] = pd.Categorical(
    df_clean["pay_band"],
    categories=pay_band_order,
    ordered=True
)

display(
    pd.DataFrame({
        "dtype": df_clean[["education_level", "pay_band"]].dtypes.astype(str),
        "ordered": [
            df_clean[col].cat.ordered
            for col in ["education_level", "pay_band"]
        ],
        "categories": [
            list(df_clean[col].cat.categories)
            for col in ["education_level", "pay_band"]
        ]
    }, index=["education_level", "pay_band"])
)

,dtype,ordered,categories
education_level,category,True,"[High School, Associate Degree, Bachelor's, Ma..."
pay_band,category,True,"[Band 1, Band 2, Band 3, Band 4, Band 5]"


Job titles can also help identify employee seniority.

I'll group them into four ordered levels:

1. **IC:** Individual contributors and specialists.
2. **Senior:** Experienced individual contributors.
3. **Manager / Lead:** Team leaders and managers.
4. **Executive:** C-level, VP, and department heads.

The classification is based on job titles. Original titles will remain unchanged.

In [5]:
job_levels = {
    "Executive": [
        "COO", "CMO", "CFO",
        "VP Design", "VP Product", "VP People", "VP Sales",
        "Head of Data", "General Counsel"
    ],
    "Manager / Lead": [
        "IT Manager", "Customer Success Manager",
        "Support Team Lead", "Design Lead",
        "Product Manager", "Operations Manager",
        "Senior Product Manager", "Marketing Manager",
        "Finance Manager", "Sales Manager",
        "Controller", "HR Manager", "Engineering Manager"
    ],
    "Senior": [
        "Senior Data Scientist", "Senior Software Engineer",
        "Staff Engineer", "Software Engineer II"
    ],
    "IC": [
        "Systems Administrator", "Paralegal",
        "Support Agent", "Legal Counsel",
        "IT Support Specialist", "Logistics Coordinator",
        "UX Designer", "Operations Analyst",
        "Product Analyst", "Product Designer",
        "HR Business Partner", "Recruiter",
        "Accountant", "HR Coordinator",
        "Sales Rep", "Content Strategist",
        "SEO Specialist", "ML Engineer",
        "Sales Development Rep", "Marketing Coordinator",
        "Data Analyst", "Account Executive",
        "Data Scientist", "Financial Analyst",
        "DevOps Engineer", "Software Engineer I",
        "QA Engineer"
    ]
}

job_level_mapping = {
    title.casefold(): level
    for level, titles in job_levels.items()
    for title in titles
}

normalized_titles = (
    df_clean["job_title"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

unknown_titles = normalized_titles[
    ~normalized_titles.isin(job_level_mapping)
]

assert unknown_titles.empty, f"Unknown job titles: {unknown_titles.unique().tolist()}"

job_level_order = [
    "IC",
    "Senior",
    "Manager / Lead",
    "Executive"
]

df_clean["job_level"] = pd.Categorical(
    normalized_titles.map(job_level_mapping),
    categories=job_level_order,
    ordered=True
)

display(
    df_clean["job_level"]
    .value_counts(sort=False)
    .to_frame("count")
)

print(f"Unique job titles: {df_clean['job_title'].nunique()}")
print(f"Job levels: {df_clean['job_level'].nunique()}")
print(f"Unclassified records: {df_clean['job_level'].isna().sum():,}")

,count
job_level,
IC,48337
Senior,4863
Manager / Lead,25108
Executive,16692


Unique job titles: 103
Job levels: 4
Unclassified records: 0


All 95,000 employees were assigned to one of the four job levels.

No unclassified records remain.

## 4. Export

Save the dataset in Parquet format to preserve data types, missing values, and ordered categories.

In [6]:
from pathlib import Path

output_path = Path("data/processed/04_data_types.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(output_path, index=False)

saved = pd.read_parquet(output_path)

pd.testing.assert_frame_equal(df_clean, saved)

print(f"Saved: {output_path}")
print(f"Rows: {len(saved):,}")
print(f"Columns: {len(saved.columns)}")
print("Data types preserved.")

Saved: data/processed/04_data_types.parquet
Rows: 95,000
Columns: 33
Data types preserved.
